In [1]:
#env var required for my ROCM version
from os import putenv
putenv("HSA_OVERRIDE_GFX_VERSION", "11.0.0") #export HSA_OVERRIDE_GFX_VERSION=11.0.0
!export HSA_OVERRIDE_GFX_VERSION=11.0.0

In [2]:
import cv2
import numpy as np
import json, random
import torch
import os
from torch.utils.data import Dataset
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import DataLoader
import torch.nn.functional as F

COLORS = ["black","brown","red","orange","yellow","green","blue","violet","gray","white","gold","silver"]
PAD = "none"
CLASSES = COLORS + [PAD]
MAX_BANDS = 6

In [4]:
import detectron2

from detectron2.utils.logger import setup_logger
setup_logger()
import matplotlib.pyplot as plt
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("resistors_train",)
cfg.DATASETS.TEST = ()
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")  # Let training initialize from model zoo
cfg.SOLVER.IMS_PER_BATCH = 2  # This is the real "batch size" commonly known to deep learning people
cfg.SOLVER.BASE_LR = 0.00025  # pick a good LR
cfg.SOLVER.MAX_ITER = 300    # 300 iterations seems good enough for this toy dataset; you will need to train longer for a practical dataset
cfg.SOLVER.STEPS = []        # do not decay learning rate
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128   # The "RoIHead batch size". 128 is faster, and good enough for this toy dataset (default: 512)
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")  
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7   
predictor = DefaultPredictor(cfg)

[01/03 20:09:02 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from ./output/model_final.pth ...


In [5]:
def coco_xywh_to_xyxy(b):
    x, y, w, h = b
    return [x, y, x + w, y + h]

def norm_color_name(s):
    # real labels are like "Red" -> "red"
    return str(s).strip().lower()

def crop_from_bbox(im, xyxy, pad=0.15):
    h, w = im.shape[:2]
    x1, y1, x2, y2 = map(float, xyxy)
    bw, bh = x2 - x1, y2 - y1
    x1 -= pad * bw; x2 += pad * bw
    y1 -= pad * bh; y2 += pad * bh
    x1 = int(max(0, x1)); y1 = int(max(0, y1))
    x2 = int(min(w - 1, x2)); y2 = int(min(h - 1, y2))
    return im[y1:y2, x1:x2].copy()

def detect_best_bbox_xyxy(predictor, im_bgr, score_thresh=0.5):
    outputs = predictor(im_bgr)
    inst = outputs["instances"].to("cpu")
    if len(inst) == 0:
        return None
    scores = inst.scores.numpy()
    best = int(scores.argmax())
    if float(scores[best]) < score_thresh:
        return None
    xyxy = inst.pred_boxes.tensor.numpy()[best]
    return xyxy.tolist()


In [6]:
def build_real_records(coco_json_path, images_dir):
    with open(coco_json_path, "r") as f:
        coco = json.load(f)

    img_by_id = {img["id"]: img for img in coco["images"]}
    records = []

    for ann in coco["annotations"]:
        if ann.get("category_id") != 1:
            continue

        img = img_by_id[ann["image_id"]]
        img_path = os.path.join(images_dir, img["file_name"])

        bands = ann.get("attributes", {}).get("colors", [])
        bands = [norm_color_name(c) for c in bands]
        if not bands:
            continue

        records.append({
            "image_path": img_path,
            "bbox_xyxy": coco_xywh_to_xyxy(ann["bbox"]),
            "bands": bands,
            "source": "real",
            "image_id": ann["image_id"],
            "ann_id": ann["id"],
        })
    return records

def build_synth_records(orders_json_path, synth_root, predictor, score_thresh=0.5):
    with open(orders_json_path, "r") as f:
        orders = json.load(f)

    records = []
    for i, o in enumerate(orders):
        img_path = os.path.join(synth_root, o["file"])
        im = cv2.imread(img_path)
        if im is None:
            continue

        bbox = detect_best_bbox_xyxy(predictor, im, score_thresh=score_thresh)
        if bbox is None:
            continue

        bands = [norm_color_name(c) for c in o["bands"]]
        records.append({
            "image_path": img_path,
            "bbox_xyxy": bbox,
            "bands": bands,
            "source": "synthetic",
            "order_idx": i
        })
    return records

real = build_real_records("dataset/real/all.json", "dataset/real/images")
synth = build_synth_records("dataset/synthetic/bands.json", "dataset/synthetic", predictor, score_thresh=0.6)

records = real + synth

with open("band_manifest.json", "w") as f:
    json.dump(records, f, indent=2)
print("real:", len(real), "synth:", len(synth), "total:", len(records))


/home/tyler/.global_venv/lib64/python3.13/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0103 20:09:09.055000 17476 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


real: 705 synth: 3000 total: 3705


In [7]:
import torch
from torch.utils.data import Dataset

def resize_letterbox(img, out_h=96, out_w=320):
    h, w = img.shape[:2]
    scale = min(out_w / w, out_h / h)
    nh, nw = int(h * scale), int(w * scale)
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((out_h, out_w, 3), dtype=np.uint8)
    y0 = (out_h - nh) // 2
    x0 = (out_w - nw) // 2
    canvas[y0:y0+nh, x0:x0+nw] = resized
    return canvas

class ResistorBandDataset(Dataset):
    def __init__(self, records, augment=True, out_h=96, out_w=320, max_bands=MAX_BANDS):
        self.records = records
        self.augment = augment
        self.out_h, self.out_w = out_h, out_w
        self.max_bands = max_bands
        self.class_to_id = {c: i for i, c in enumerate(CLASSES)}
        self.pad_id = self.class_to_id[PAD]

    def encode(self, bands):
        ids = [self.class_to_id[norm_color_name(b)] for b in bands]
        ids = ids[:self.max_bands]
        ids += [self.pad_id] * (self.max_bands - len(ids))
        return torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        im = cv2.imread(r["image_path"])
        crop = crop_from_bbox(im, r["bbox_xyxy"], pad=0.15)
        crop = resize_letterbox(crop, self.out_h, self.out_w)

        bands = list(r["bands"])

        if self.augment and np.random.rand() < 0.2:
            crop = crop[:, ::-1, :].copy()
            bands = bands[::-1]

        # photometric adj
        if self.augment:
            if np.random.rand() < 0.7:
                alpha = 1.0 + np.random.uniform(-0.15, 0.15)  # contrast
                beta = np.random.uniform(-12, 12)             # brightness
                crop = np.clip(alpha * crop + beta, 0, 255).astype(np.uint8)

        x = torch.from_numpy(crop).permute(2, 0, 1).float() / 255.0
        y = self.encode(bands)
        return x, y


In [8]:

class BandNet(nn.Module):
    def __init__(self, num_classes=len(CLASSES), max_bands=MAX_BANDS):
        super().__init__()
        self.max_bands = max_bands
        self.num_classes = num_classes

        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.head = nn.Linear(512, max_bands * num_classes)

    def forward(self, x):
        f = self.backbone(x)  # [B,512]
        logits = self.head(f).view(-1, self.max_bands, self.num_classes)
        return logits

def band_loss(logits, target_ids):
    B, M, C = logits.shape
    return nn.CrossEntropyLoss()(logits.reshape(B*M, C), target_ids.reshape(B*M))


In [9]:
with open("band_manifest.json","r") as f:
    records = json.load(f)

real = [r for r in records if r["source"] == "real"]
synth = [r for r in records if r["source"] == "synthetic"]

mixed = synth + real * 3 # oversample real data
random.shuffle(mixed)

ds = ResistorBandDataset(mixed, augment=True)
dl = DataLoader(ds, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = BandNet().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

for epoch in range(20):
    model.train()
    total = 0.0
    for x, y in dl:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = band_loss(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += float(loss.item())
    print("epoch", epoch, "loss", total/len(dl))

torch.save(model.state_dict(), "bandnet.pt")


epoch 0 loss 1.5049377549439669
epoch 1 loss 1.0413787193596362
epoch 2 loss 0.805215559899807
epoch 3 loss 0.6287702897563576
epoch 4 loss 0.5087479470297694
epoch 5 loss 0.40359610728919504
epoch 6 loss 0.31569182109087707
epoch 7 loss 0.2524326253216714
epoch 8 loss 0.21779331765137613
epoch 9 loss 0.18001200770959258
epoch 10 loss 0.1492890309309587
epoch 11 loss 0.1198971769073978
epoch 12 loss 0.10236529821995646
epoch 13 loss 0.08344106111908331
epoch 14 loss 0.07761968799168244
epoch 15 loss 0.07867805786663666
epoch 16 loss 0.09093292409088463
epoch 17 loss 0.07197001951280982
epoch 18 loss 0.05522884370875545
epoch 19 loss 0.045345018606167284


In [10]:
def decode_bands(logits):
    # logits: [1, MAX_BANDS, C]
    probs = F.softmax(logits, dim=-1)[0]             # [M,C]
    ids = probs.argmax(dim=-1).tolist()
    conf = probs.max(dim=-1).values.tolist()
    bands = []
    for i, cid in enumerate(ids):
        name = CLASSES[cid]
        if name == PAD:
            break
        bands.append((name, conf[i]))
    return bands

def predict_bands_for_crop(model, crop_bgr, device):
    # prepare two versions: normal + flipped
    crop1 = resize_letterbox(crop_bgr, 96, 320)
    crop2 = crop1[:, ::-1, :].copy()

    x1 = torch.from_numpy(crop1).permute(2,0,1).float().unsqueeze(0)/255.0
    x2 = torch.from_numpy(crop2).permute(2,0,1).float().unsqueeze(0)/255.0
    x1, x2 = x1.to(device), x2.to(device)

    with torch.no_grad():
        l1 = model(x1)
        l2 = model(x2)

    b1 = decode_bands(l1)
    b2 = decode_bands(l2)
    # reverse label order back for flipped prediction
    b2 = list(reversed(b2))

    # choose higher mean confidence
    s1 = np.mean([c for _,c in b1]) if b1 else 0.0
    s2 = np.mean([c for _,c in b2]) if b2 else 0.0
    return b1 if s1 >= s2 else b2
